In [26]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np

In [27]:
df = pd.read_csv("lightcast_job_postings.csv", low_memory=False)
print(f"Total postings: {len(df):,}")
print(df.columns.tolist())

Total postings: 72,498
['ID', 'LAST_UPDATED_DATE', 'LAST_UPDATED_TIMESTAMP', 'DUPLICATES', 'POSTED', 'EXPIRED', 'DURATION', 'SOURCE_TYPES', 'SOURCES', 'URL', 'ACTIVE_URLS', 'ACTIVE_SOURCES_INFO', 'TITLE_RAW', 'BODY', 'MODELED_EXPIRED', 'MODELED_DURATION', 'COMPANY', 'COMPANY_NAME', 'COMPANY_RAW', 'COMPANY_IS_STAFFING', 'EDUCATION_LEVELS', 'EDUCATION_LEVELS_NAME', 'MIN_EDULEVELS', 'MIN_EDULEVELS_NAME', 'MAX_EDULEVELS', 'MAX_EDULEVELS_NAME', 'EMPLOYMENT_TYPE', 'EMPLOYMENT_TYPE_NAME', 'MIN_YEARS_EXPERIENCE', 'MAX_YEARS_EXPERIENCE', 'IS_INTERNSHIP', 'SALARY', 'REMOTE_TYPE', 'REMOTE_TYPE_NAME', 'ORIGINAL_PAY_PERIOD', 'SALARY_TO', 'SALARY_FROM', 'LOCATION', 'CITY', 'CITY_NAME', 'COUNTY', 'COUNTY_NAME', 'MSA', 'MSA_NAME', 'STATE', 'STATE_NAME', 'COUNTY_OUTGOING', 'COUNTY_NAME_OUTGOING', 'COUNTY_INCOMING', 'COUNTY_NAME_INCOMING', 'MSA_OUTGOING', 'MSA_NAME_OUTGOING', 'MSA_INCOMING', 'MSA_NAME_INCOMING', 'NAICS2', 'NAICS2_NAME', 'NAICS3', 'NAICS3_NAME', 'NAICS4', 'NAICS4_NAME', 'NAICS5', 'NAICS5

In [28]:
needed = ['NAICS_2022_6_NAME', 'REMOTE_TYPE_NAME']
df_remote = df[needed].copy()
df_remote = df_remote[df_remote['NAICS_2022_6_NAME'].notna()].copy()

print(df_remote.head())
print(df_remote.shape)

                            NAICS_2022_6_NAME REMOTE_TYPE_NAME
0  Automotive Parts and Accessories Retailers           [None]
1                     Temporary Help Services           Remote
2                            Claims Adjusting           [None]
3                          Commercial Banking           [None]
4                       Unclassified Industry           [None]
(72454, 2)


In [29]:
top_industries = (
    df_remote['NAICS_2022_6_NAME']
    .value_counts()
    .head(15)
    .index
)

print(top_industries)

Index(['Unclassified Industry', 'Custom Computer Programming Services',
       'Administrative Management and General Management Consulting Services',
       'Employment Placement Agencies', 'Computer Systems Design Services',
       'Commercial Banking', 'Offices of Certified Public Accountants',
       'Temporary Help Services',
       'Direct Health and Medical Insurance Carriers',
       'Other Computer Related Services',
       'Colleges, Universities, and Professional Schools',
       'General Medical and Surgical Hospitals',
       'Other Scientific and Technical Consulting Services',
       'Software Publishers',
       'All Other Professional, Scientific, and Technical Services'],
      dtype='object', name='NAICS_2022_6_NAME')


In [31]:
df['POSTED'].head(10)

0    6/2/2024
1    6/2/2024
2    6/2/2024
3    6/2/2024
4    6/2/2024
5    6/2/2024
6    6/2/2024
7    6/2/2024
8    6/2/2024
9    6/2/2024
Name: POSTED, dtype: object

In [32]:
df_remote_time = df[['NAICS_2022_6_NAME', 'REMOTE_TYPE_NAME', 'POSTED']].copy()
df_remote_time = df_remote_time[df_remote_time['NAICS_2022_6_NAME'].notna()].copy()

df_remote_time['POSTED'] = pd.to_datetime(df_remote_time['POSTED'], errors='coerce')
df_remote_time['month'] = df_remote_time['POSTED'].dt.to_period('M')

print(df_remote_time[['POSTED', 'month']].head())
print(df_remote_time['month'].value_counts().sort_index())

      POSTED    month
0 2024-06-02  2024-06
1 2024-06-02  2024-06
2 2024-06-02  2024-06
3 2024-06-02  2024-06
4 2024-06-02  2024-06
month
2024-05    15133
2024-06    14381
2024-07    12502
2024-08    15105
2024-09    15333
Freq: M, Name: count, dtype: int64


In [33]:
def classify_remote(x):
    x = str(x).lower()
    if 'remote' in x:
        return 'Remote'
    else:
        return 'Onsite'

df_remote_time['remote_group'] = df_remote_time['REMOTE_TYPE_NAME'].apply(classify_remote)

print(df_remote_time['remote_group'].value_counts())

remote_group
Onsite    56570
Remote    15884
Name: count, dtype: int64


In [34]:
df_time_top = df_remote_time[df_remote_time['NAICS_2022_6_NAME'].isin(top_industries)].copy()

monthly_counts = (
    df_time_top.groupby(['month', 'NAICS_2022_6_NAME', 'remote_group'])
    .size()
    .reset_index(name='count')
)

monthly_totals = (
    df_time_top.groupby(['month', 'NAICS_2022_6_NAME'])
    .size()
    .reset_index(name='industry_month_total')
)

monthly_summary = monthly_counts.merge(
    monthly_totals,
    on=['month', 'NAICS_2022_6_NAME'],
    how='left'
)

monthly_summary['share_pct'] = (
    monthly_summary['count'] / monthly_summary['industry_month_total'] * 100
)

monthly_summary.head(20)

,month,NAICS_2022_6_NAME,remote_group,count,industry_month_total,share_pct
0,2024-05,Administrative Management and General Manageme...,Onsite,1158,1224,94.607843
1,2024-05,Administrative Management and General Manageme...,Remote,66,1224,5.392157
2,2024-05,"All Other Professional, Scientific, and Techni...",Onsite,158,209,75.598086
3,2024-05,"All Other Professional, Scientific, and Techni...",Remote,51,209,24.401914
4,2024-05,"Colleges, Universities, and Professional Schools",Onsite,203,259,78.378378
5,2024-05,"Colleges, Universities, and Professional Schools",Remote,56,259,21.621622
6,2024-05,Commercial Banking,Onsite,371,426,87.089202
7,2024-05,Commercial Banking,Remote,55,426,12.910798
8,2024-05,Computer Systems Design Services,Onsite,779,945,82.433862
9,2024-05,Computer Systems Design Services,Remote,166,945,17.566138


In [22]:
monthly_remote = monthly_summary[monthly_summary['remote_group'] == 'Remote'].copy()
monthly_remote = monthly_remote.sort_values(['NAICS_2022_6_NAME', 'month'])

monthly_remote[['month', 'NAICS_2022_6_NAME', 'share_pct']].head(30)

,month,NAICS_2022_6_NAME,share_pct
1,2024-05,Administrative Management and General Manageme...,5.392157
31,2024-06,Administrative Management and General Manageme...,9.090909
61,2024-07,Administrative Management and General Manageme...,12.441680
91,2024-08,Administrative Management and General Manageme...,21.725572
121,2024-09,Administrative Management and General Manageme...,8.514493
3,2024-05,"All Other Professional, Scientific, and Techni...",24.401914
33,2024-06,"All Other Professional, Scientific, and Techni...",22.352941
63,2024-07,"All Other Professional, Scientific, and Techni...",28.638498
93,2024-08,"All Other Professional, Scientific, and Techni...",32.894737
123,2024-09,"All Other Professional, Scientific, and Techni...",29.186603


In [35]:
trend_table = monthly_remote.pivot(
    index='NAICS_2022_6_NAME',
    columns='month',
    values='share_pct'
)

trend_table['change_may_to_sep'] = trend_table['2024-09'] - trend_table['2024-05']
trend_table = trend_table.sort_values('change_may_to_sep', ascending=False)

trend_table

month,2024-05,2024-06,2024-07,2024-08,2024-09,change_may_to_sep
NAICS_2022_6_NAME,,,,,,
Unclassified Industry,17.524339,19.308832,22.887768,23.535792,28.922040,11.397701
Employment Placement Agencies,17.358892,18.463612,27.870968,33.362754,25.611052,8.252160
General Medical and Surgical Hospitals,17.889908,15.508021,20.408163,15.000000,24.125874,6.235966
"All Other Professional, Scientific, and Technical Services",24.401914,22.352941,28.638498,32.894737,29.186603,4.784689
Administrative Management and General Management Consulting Services,5.392157,9.090909,12.441680,21.725572,8.514493,3.122336
"Colleges, Universities, and Professional Schools",21.621622,22.314050,20.141343,26.148410,21.843003,0.221382
Other Scientific and Technical Consulting Services,31.439394,24.623116,22.613065,31.531532,31.443299,0.003905
Offices of Certified Public Accountants,3.619910,16.265060,20.912548,2.282158,2.477477,-1.142432
Temporary Help Services,34.177215,21.064815,41.401274,21.739130,31.666667,-2.510549


In [36]:
import matplotlib.pyplot as plt

change_plot = trend_table[['change_may_to_sep']].sort_values('change_may_to_sep', ascending=True)

plt.figure(figsize=(12, 8))
plt.barh(change_plot.index, change_plot['change_may_to_sep'])
plt.xlabel('Change in Remote Share (Sep 2024 - May 2024, percentage points)')
plt.ylabel('Industry')
plt.title('Change in Remote Job Share by Industry (May to September 2024)')
plt.tight_layout()
plt.savefig("change_in_remote_share_may_to_sep.png", dpi=150, bbox_inches='tight')
print("saved")

saved
